In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 5: residual model with leave-one-block-out scoring

Target: `ground_temp_k - baseline_k` per REMS row. Prediction = baseline + residual model.

- **No leakage from the baseline fit.** In fold b, the baseline for every row (training and held out) comes from the fold-b parameters in `fit_params.csv`, fitted without block b.
- **Learner:** scikit-learn `HistGradientBoostingRegressor`, settings fixed in `residual.py` before the first run and not tuned.
- **Training rows:** the fold's training blocks, without placeholder-tau sols and sols 3763-3778 (same exclusions as the baseline fit). Both groups are scored separately.
- **Benchmark:** mean REMS per (LTST bin, 30-degree Ls bin) from the fold's training rows, as in Step 4.

## Ablation and decision rule (fixed before running)

Four configurations, one learner setting: 4 configurations tried in total.

| Config | Predictors |
|---|---|
| `time_only` | sin/cos LTST, sin/cos Ls, tau, baseline_k |
| `no_elev` | time_only + TI, slope, sin/cos aspect, albedo |
| `full` | no_elev + elevation |
| `no_elev_flux` | no_elev + the model's absorbed flux at the bin centre |

Headline metric: macro RMSE over the 9 training blocks (placeholder-tau rows excluded). Applied in order:

1. **Static covariates:** choose `no_elev` if its macro RMSE is at least 0.1 K below `time_only`; otherwise `time_only`.
2. **Elevation** (it can act as a campaign ID): switch to `full` only if `no_elev` was chosen, `full` is at least 0.2 K better in macro RMSE, and better in at least 6 of 9 blocks.
3. **Flux term:** switch to `no_elev_flux` only if `no_elev` is still chosen, it is at least 0.1 K better, and better in at least 5 of 9 blocks.
4. **Second learner (xgboost):** only if the chosen configuration is within 0.3 K of the benchmark macro RMSE either way. Not run in this notebook.

Spatial skill cannot be measured here: all rover positions lie within about 10 x 5 km, and every score below is a time-and-place-along-the-traverse score.

In [ ]:
# --- config ---
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r

SUSPECT_SOLS = (3763, 3778)
EXPECTED_TRAIN_BLOCKS = [0, 1, 2, 3, 5, 6, 7, 9, 10]
LS_BIN_DEG = 30
STATIC_MIN_GAIN_K = 0.1
ELEV_MIN_GAIN_K, ELEV_MIN_BLOCK_WINS = 0.2, 6
FLUX_MIN_GAIN_K, FLUX_MIN_BLOCK_WINS = 0.1, 5
SECOND_LEARNER_WINDOW_K = 0.3
REPRO_TOL_K = 0.05          # fold baselines vs Step 4 lobo_rows (spin-up tolerance is 0.01 K)
PDP_POINTS = 25
FIT_PARAMS = paths.OUT_DIR / "fit_params.csv"
STEP4_ROWS = paths.OUT_DIR / "lobo_rows.csv"
OUT = paths.RESIDUAL_OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = OUT / "residual_progress.log"
PROGRESS_LOG.write_text("")
CONFIGS = r.CONFIGS
print("configs:", {k: len(v) for k, v in CONFIGS.items()})

## Data and fold baselines

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
blocks = sorted(tbl.block.unique())
assert blocks == EXPECTED_TRAIN_BLOCKS, f"training blocks {blocks}"

per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "slope_deg", "aspect_deg",
                "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
row_step = t.bin_centre_step(tbl.bin.to_numpy())
evaluate = t.make_evaluator(sol_inputs, row_sol, row_step)

obs = tbl.ground_temp_k.to_numpy()
block = tbl.block.to_numpy()
suspect = tbl.sol.between(*SUSPECT_SOLS).to_numpy()
placeholder = tbl.tau_is_placeholder.to_numpy()
fit_ok = ~suspect & ~placeholder
tbl["group"] = np.where(suspect, "suspect_terrain", np.where(placeholder, "placeholder_tau", "main"))
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ltst_hr"] = (tbl.bin + 0.5) * t.BIN_HOURS
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
print(len(sols), "sols,", len(tbl), "rows;", int(fit_ok.sum()), "rows usable for training")
print(tbl.group.value_counts().to_string())

fits = pd.read_csv(FIT_PARAMS).set_index("fold")
assert list(fits.index[:-1]) == [f"block {b}" for b in blocks] and fits.index[-1] == "all training blocks"

In [ ]:
def fold_inputs(p):
    """Baseline (K) and absorbed flux at the bin centre (W m^-2) for every row, at parameters p."""
    g_diff, f_ir, _ = p
    q = t.absorbed_flux(sol_inputs["lat_deg"], sol_inputs["ls_deg"], sol_inputs["tau"],
                        sol_inputs["slope_deg"], sol_inputs["aspect_deg"], sol_inputs["albedo"],
                        t.step_ltst(), g_diff=g_diff, f_ir=f_ir)
    return evaluate(p).copy(), q[row_sol, row_step].copy()


fold_base, fold_q = {}, {}
t0 = time.time()
for label in fits.index:
    p = tuple(fits.loc[label, list(t.PARAM_NAMES)].to_numpy(dtype=float))
    fold_base[label], fold_q[label] = fold_inputs(p)
    assert np.all(np.isfinite(fold_base[label])), f"non-finite baseline in {label}"
    line = f"{label}: params {np.round(p, 4)} | {time.time() - t0:.0f} s"
    print(line)
    with open(PROGRESS_LOG, "a") as fh:
        fh.write(line + "\n")

# Reproduce Step 4's out-of-fold baseline for each held-out block.
step4 = pd.read_csv(STEP4_ROWS)
assert (step4.sol.to_numpy() == tbl.sol.to_numpy()).all() and (step4.bin.to_numpy() == tbl.bin.to_numpy()).all(), \
    "row order differs from Step 4"
oof = np.full(len(tbl), np.nan)
for b in blocks:
    oof[block == b] = fold_base[f"block {b}"][block == b]
diff = np.abs(oof - step4.baseline_k.to_numpy())
print(f"max |fold baseline - Step 4 baseline_k| = {diff.max():.4f} K")
assert diff.max() < REPRO_TOL_K, "fold baselines do not reproduce Step 4"

## LOBO: residual model per fold and configuration

In [ ]:
tbl["baseline_k"] = oof
tbl["bench_k"] = np.nan
for c in CONFIGS:
    tbl[f"pred_{c}"] = np.nan
n_fallback = {}
t0 = time.time()
for b in blocks:
    label = f"block {b}"
    held = block == b
    train = fit_ok & ~held
    frame = tbl.assign(baseline_k=fold_base[label], q_abs_wm2=fold_q[label])
    X = r.make_features(frame)
    target = obs - fold_base[label]
    for c, cols in CONFIGS.items():
        model = r.make_learner().fit(X.loc[train, cols], target[train])
        tbl.loc[held, f"pred_{c}"] = fold_base[label][held] + model.predict(X.loc[held, cols])
    bench, n_fallback[b] = r.benchmark(tbl[~held], tbl[held])
    tbl.loc[held, "bench_k"] = bench
    line = f"fold {b:2d}: {int(train.sum())} training rows | benchmark fallback rows {n_fallback[b]} | {time.time() - t0:.0f} s"
    print(line)
    with open(PROGRESS_LOG, "a") as fh:
        fh.write(line + "\n")
assert tbl.filter(like="pred_").notna().all().all() and tbl.bench_k.notna().all()
step4_bench_diff = np.abs(tbl.bench_k.to_numpy() - step4.bench_k.to_numpy()).max()
print(f"max |benchmark - Step 4 bench_k| = {step4_bench_diff:.2e} K")
assert step4_bench_diff < 1e-9, "benchmark differs from Step 4"

## Scores and the decision rule

In [ ]:
MODELS = ["baseline", "bench"] + list(CONFIGS)
COLS = {"baseline": "baseline_k", "bench": "bench_k", **{c: f"pred_{c}" for c in CONFIGS}}
for m in MODELS:
    tbl[f"resid_{m}"] = tbl[COLS[m]] - tbl.ground_temp_k


def score(g, models=MODELS):
    out = {"n": len(g)}
    for m in models:
        out[f"{m}_rmse_k"] = np.sqrt((g[f"resid_{m}"] ** 2).mean())
        out[f"{m}_bias_k"] = g[f"resid_{m}"].mean()
    return pd.Series(out)


scored = tbl[tbl.group != "placeholder_tau"]        # block 10 is all suspect_terrain and is its own block
main = tbl[tbl.group == "main"]
by_block = scored.groupby("block").apply(score, include_groups=False)
by_group = tbl.groupby("group").apply(score, include_groups=False)

summary = pd.DataFrame({
    "macro_rmse_k": {m: by_block[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_3763_k": {m: by_block.drop(index=10)[f"{m}_rmse_k"].mean() for m in MODELS},
    "pooled_rmse_main_k": {m: np.sqrt((main[f"resid_{m}"] ** 2).mean()) for m in MODELS},
    "rmse_placeholder_tau_k": {m: by_group.loc["placeholder_tau", f"{m}_rmse_k"] for m in MODELS},
    "rmse_suspect_terrain_k": {m: by_group.loc["suspect_terrain", f"{m}_rmse_k"] for m in MODELS},
    "blocks_beating_bench": {m: int((by_block[f"{m}_rmse_k"] < by_block.bench_rmse_k).sum()) for m in MODELS},
})
pd.options.display.float_format = "{:.2f}".format
print("Summary (placeholder-tau rows excluded from block scores):")
print(summary.to_string())
print()
print("Per-block RMSE (K):")
print(by_block[["n"] + [f"{m}_rmse_k" for m in MODELS]].to_string())
print()
print("Per-block bias (K):")
print(by_block[[f"{m}_bias_k" for m in MODELS]].to_string())

In [ ]:
macro = summary.macro_rmse_k


def wins(a, b):
    return int((by_block[f"{a}_rmse_k"] < by_block[f"{b}_rmse_k"]).sum())


log = []
chosen = "no_elev" if macro["no_elev"] <= macro["time_only"] - STATIC_MIN_GAIN_K else "time_only"
log.append(f"1. static covariates: no_elev {macro['no_elev']:.3f} K vs time_only {macro['time_only']:.3f} K -> {chosen}")
if chosen == "no_elev":
    gain, w = macro["no_elev"] - macro["full"], wins("full", "no_elev")
    if gain >= ELEV_MIN_GAIN_K and w >= ELEV_MIN_BLOCK_WINS:
        chosen = "full"
    log.append(f"2. elevation: gain {gain:.3f} K, better in {w}/9 blocks -> {chosen}")
else:
    log.append("2. elevation: not considered (static covariates not adopted)")
if chosen == "no_elev":
    gain, w = macro["no_elev"] - macro["no_elev_flux"], wins("no_elev_flux", "no_elev")
    if gain >= FLUX_MIN_GAIN_K and w >= FLUX_MIN_BLOCK_WINS:
        chosen = "no_elev_flux"
    log.append(f"3. flux term: gain {gain:.3f} K, better in {w}/9 blocks -> {chosen}")
else:
    log.append(f"3. flux term: not considered (chosen is {chosen})")
gap = macro[chosen] - macro["bench"]
second = abs(gap) <= SECOND_LEARNER_WINDOW_K
log.append(f"4. chosen {chosen}: macro {macro[chosen]:.3f} K vs benchmark {macro['bench']:.3f} K "
           f"(gap {gap:+.3f} K) -> second learner {'justified' if second else 'not justified'}")
print("\n".join(log))
(OUT / "residual_decision.txt").write_text("\n".join(log) + "\n")

In [ ]:
show = ["baseline", "bench", chosen]
by_hour = main.groupby("ltst_hour").apply(score, models=show, include_groups=False)
by_ls = main.groupby("ls_bin").apply(score, models=show, include_groups=False)
print(f"By LTST hour (main rows), chosen = {chosen}:")
print(by_hour.to_string())
print()
print("By Ls bin (main rows):")
print(by_ls[["n"] + [f"{m}_bias_k" for m in show] + [f"{m}_rmse_k" for m in show]].to_string())

tbl.to_csv(OUT / "residual_rows.csv", index=False)
summary.to_csv(OUT / "residual_summary.csv")
by_block.to_csv(OUT / "residual_by_block.csv")
by_group.to_csv(OUT / "residual_by_group.csv")
by_hour.to_csv(OUT / "residual_by_ltst_hour.csv")
by_ls.to_csv(OUT / "residual_by_ls_bin.csv")

## Covariate effects (partial dependence)

The residual model must not undo the baseline's spatial terms on the map. Partial dependence of the residual on each static covariate, from a model trained on all training rows (all-blocks baseline parameters). Shown for `no_elev` and `full` whatever was chosen, so the size of each effect is visible. Ranges are over the rover's covariate values only; the map will extrapolate beyond them.

In [ ]:
label = "all training blocks"
frame_all = tbl.assign(baseline_k=fold_base[label], q_abs_wm2=fold_q[label])
X_all = r.make_features(frame_all)
target_all = obs - fold_base[label]
pdp_specs = {"ti_themis_tiu": "ti_themis_tiu", "slope_deg": "slope_deg", "aspect_deg": "aspect_deg",
             "albedo_mean": "albedo_mean", "elev_dem_m": "elev_dem_m"}
pdp_rows = []
fig, axes = plt.subplots(1, len(pdp_specs), figsize=(18, 3.5))
for c in ["no_elev", "full"]:
    cols = CONFIGS[c]
    model = r.make_learner().fit(X_all.loc[fit_ok, cols], target_all[fit_ok])
    base_rows = frame_all[fit_ok]
    for ax, (name, col) in zip(axes, pdp_specs.items()):
        if name == "elev_dem_m" and c == "no_elev":
            continue
        lo, hi = (0.0, 360.0) if name == "aspect_deg" else np.percentile(base_rows[col], [1, 99])
        grid = np.linspace(lo, hi, PDP_POINTS)
        curve = np.array([model.predict(r.make_features(base_rows.assign(**{col: v}))[cols]).mean() for v in grid])
        ax.plot(grid, curve, "-o", ms=3, label=c)
        pdp_rows.append({"config": c, "covariate": name, "range_k": curve.max() - curve.min(),
                         "lo": lo, "hi": hi})
        ax.set(xlabel=name, title=name)
axes[0].set_ylabel("mean predicted residual (K)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "residual_pdp.png", dpi=120)
pdp = pd.DataFrame(pdp_rows)
pdp.to_csv(OUT / "residual_pdp.csv", index=False)
print("Partial-dependence range (max - min of the mean predicted residual), K:")
print(pdp.pivot(index="covariate", columns="config", values="range_k").to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ax = axes[0]
x = np.arange(len(by_block))
for i, m in enumerate(show):
    ax.bar(x + (i - 1) * 0.27, by_block[f"{m}_rmse_k"], 0.27, label=m)
ax.set_xticks(x, [str(b) for b in by_block.index])
ax.set(xlabel="held-out block", ylabel="RMSE (K)", title="Leave-one-block-out RMSE")
ax.legend(fontsize=8)
ax = axes[1]
for m, st in zip(show, ["o-", "s-", "^-"]):
    ax.plot(by_hour.index, by_hour[f"{m}_bias_k"], st, ms=3, label=f"{m} bias")
    ax.plot(by_hour.index, by_hour[f"{m}_rmse_k"], st[0] + "--", ms=2, alpha=0.6, label=f"{m} RMSE")
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="LTST hour", ylabel="K", title="By LTST hour (main rows)")
ax.legend(fontsize=7)
ax = axes[2]
for i, m in enumerate(show):
    ax.bar(by_ls.index + LS_BIN_DEG / 2 + (i - 1) * 8, by_ls[f"{m}_bias_k"], 8, label=m)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="Ls (deg)", ylabel="bias (K)", title="Bias by 30-degree Ls bin (main rows)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "residual_scores.png", dpi=120)

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True, sharey=True)
for ax, b in zip(axes.flat, blocks):
    g = tbl[tbl.block == b].groupby("ltst_hour")[["ground_temp_k", "baseline_k", "bench_k", f"pred_{chosen}"]].mean()
    ax.plot(g.index, g.ground_temp_k, "k.-", label="REMS")
    ax.plot(g.index, g.baseline_k, "r.-", label="baseline (fold fit)")
    ax.plot(g.index, g.bench_k, "b.--", label="benchmark")
    ax.plot(g.index, g[f"pred_{chosen}"], "g.-", label=f"baseline + residual ({chosen})")
    s = tbl[tbl.block == b].sol
    ax.set_title(f"block {b}: sols {s.min()}-{s.max()} (held out)", fontsize=9)
axes.flat[0].legend(fontsize=7)
fig.supxlabel("LTST hour")
fig.supylabel("ground temperature (K)")
fig.tight_layout()
fig.savefig(OUT / "residual_diurnal_by_block.png", dpi=120)
print("saved to", OUT)